<a href="https://colab.research.google.com/github/dkkpd/NLP-with-disaster-tweets-kaggle-competition/blob/main/Kaggle_NLP_Disaster_Tweets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!pip install transformers datasets torch scikit-learn

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split

train = pd.read_csv("train.csv")

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train["text"].tolist(), train["target"].tolist(), test_size=0.2, random_state=42
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

Training samples: 6090
Validation samples: 1523


In [26]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

print(train_encodings["input_ids"][0])
print(tokenizer.decode(train_encodings["input_ids"][0]))

[101, 26103, 1998, 7481, 4106, 1997, 2342, 2000, 2224, 9593, 5968, 1999, 3386, 1012, 1001, 20168, 19841, 2887, 2510, 4188, 7806, 1012, 16770, 1024, 1013, 1013, 1056, 1012, 2522, 1013, 1058, 14227, 3723, 25856, 2102, 16523, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[CLS] courageous and honest analysis of need to use atomic bomb in 1945. # hiroshima70 japanese military refused surrender. https : / / t. co / vhmtytptgr [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [27]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [28]:
# Wrap data in PyTorch dataset object
import torch

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TweetDataset(train_encodings, train_labels)
val_dataset = TweetDataset(val_encodings, val_labels)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")

Train dataset size: 6090
Val dataset size: 1523


In [29]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

In [30]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [31]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.419125,0.384873,0.837163,0.803175
2,0.325659,0.388372,0.847012,0.807914
3,0.249815,0.430742,0.839133,0.807238


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1143, training_loss=0.34286784875424203, metrics={'train_runtime': 219.1032, 'train_samples_per_second': 83.385, 'train_steps_per_second': 5.217, 'total_flos': 397060678455840.0, 'train_loss': 0.34286784875424203, 'epoch': 3.0})

In [36]:
# Create a free account at huggingface.co first, then get an access token from
# huggingface.co/settings/tokens

from huggingface_hub import login
from google.colab import userdata
HF_KEY = userdata.get('HF_TOKEN').strip()
login(HF_KEY)

trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")

model.push_to_hub("dkkpd/disaster-tweets-distilbert")
tokenizer.push_to_hub("dkkpd/disaster-tweets-distilbert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/dkkpd/disaster-tweets-distilbert/commit/35b1160e00c4c84f75670ca00bc9904f9b8efeaf', commit_message='Upload tokenizer', commit_description='', oid='35b1160e00c4c84f75670ca00bc9904f9b8efeaf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/dkkpd/disaster-tweets-distilbert', endpoint='https://huggingface.co', repo_type='model', repo_id='dkkpd/disaster-tweets-distilbert'), pr_revision=None, pr_num=None)

In [37]:
# Tweaking to improve the model
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification

# Reload a fresh model
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

training_args_v2 = TrainingArguments(
    output_dir="./results_v2",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,   # changed from 2e-5
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer_v2 = Trainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer_v2.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.435091,0.385266,0.838477,0.801613
2,0.309033,0.411554,0.848982,0.810855
3,0.184474,0.520762,0.836507,0.805317


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1143, training_loss=0.3124147091429169, metrics={'train_runtime': 219.9157, 'train_samples_per_second': 83.077, 'train_steps_per_second': 5.197, 'total_flos': 397060678455840.0, 'train_loss': 0.3124147091429169, 'epoch': 3.0})

In [40]:
# Tweaking to improve the model
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification

# Reload a fresh model
model_v4 = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

training_args_v4 = TrainingArguments(
    output_dir="./results_v4",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,   # changed from 2e-5
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer_v4 = Trainer(
    model=model_v4,
    args=training_args_v4,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer_v4.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.415716,0.400865,0.836507,0.807126
2,0.351458,0.388845,0.848326,0.814458
3,0.325118,0.397169,0.845043,0.812401


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1143, training_loss=0.37742907773582746, metrics={'train_runtime': 230.7007, 'train_samples_per_second': 79.194, 'train_steps_per_second': 4.954, 'total_flos': 397060678455840.0, 'train_loss': 0.37742907773582746, 'epoch': 3.0})

In [42]:
# Save locally first
trainer_v4.save_model("./final_model_v4")   # replace `trainer` with whatever variable name holds your v4 Trainer
tokenizer.save_pretrained("./final_model_v4")

# Push to the same repo as before, overwriting with the improved version
model.push_to_hub("dkkpd/disaster-tweets-distilbert")
tokenizer.push_to_hub("dkkpd/disaster-tweets-distilbert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/dkkpd/disaster-tweets-distilbert/commit/35b1160e00c4c84f75670ca00bc9904f9b8efeaf', commit_message='Upload tokenizer', commit_description='', oid='35b1160e00c4c84f75670ca00bc9904f9b8efeaf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/dkkpd/disaster-tweets-distilbert', endpoint='https://huggingface.co', repo_type='model', repo_id='dkkpd/disaster-tweets-distilbert'), pr_revision=None, pr_num=None)